# Validación cruzada (k-fold) — BETO baseline (cabeza congelada)
Explora los learning rates más estables vistos en el grid search,
para confirmar con varios folds que el resultado no depende del split particular.

In [ ]:
# TODO = AGREGAR SIN HASHTAGS
!git clone https://github.com/camistrika/BETO_HUMOR.git
%cd BETO_HUMOR
!pip install -e . -q

In [ ]:
!pip install -q torchao --upgrade
!pip install -q transformers peft datasets scikit-learn pyyaml

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from transformers import AutoTokenizer

from betohumor.config import DataConfig, BetoConfig
from betohumor.utils import set_seed
from betohumor.dataset import load_and_split
from betohumor.models.beto import build_beto
from betohumor.hyperparam_search import run_one

/opt/miniconda3/envs/AP_FINAL/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Datos y configuraciones

In [2]:
data_config  = DataConfig(data_path = "../data/raw/haha_2019_train.csv")
beto_config = BetoConfig()
set_seed(data_config.seed)

df_train, df_val, df_test = load_and_split(data_config)

df_full = pd.concat([df_train, df_val]).reset_index(drop=True)

tokenizer = AutoTokenizer.from_pretrained(beto_config.base_model)
label_col = data_config.label_col

Train: 19197 | Val: 2397 | Test: 2400


## 2. Configuración de la búsqueda
El baseline congelado no usa LoraConfig: la única variable a explorar
es el learning rate.

In [3]:
LR_VALUES = [1e-4, 5e-4]
N_FOLDS   = 5

params_grid = [{'learning_rate': lr} for lr in LR_VALUES]


# LR_VALUES = [1e-4, 5e-4]
# WEIGHT_DECAYS = [0.01, 0.05]
# N_FOLDS = 5

# params_grid = [
#     {'learning_rate': lr, 'weight_decay': wd}
#     for lr, wd in product(LR_VALUES, WEIGHT_DECAYS)
# ]

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=data_config.seed)
params_grid




[{'learning_rate': 0.0001}, {'learning_rate': 0.0005}]

## 3. Correr la validación cruzada

In [4]:
all_fold_results = []

for params in params_grid:
    fold_scores = []
    for fold_i, (train_idx, val_idx) in enumerate(skf.split(df_full, df_full[label_col])):
        run_name = f"lr{params['learning_rate']}_fold{fold_i+1}"
        print(f"\n--- {run_name} ---")

        df_fold_train = df_full.iloc[train_idx].reset_index(drop=True)
        df_fold_val   = df_full.iloc[val_idx].reset_index(drop=True)

        output_dir = f"results/checkpoints/cv_baseline/{run_name}"
        macro_f1, trainer = run_one(
            lambda params: build_beto(beto_config), params, beto_config, data_config,
            df_fold_train, df_fold_val, tokenizer, output_dir, seed=data_config.seed,
        )

        fold_scores.append(macro_f1)
        all_fold_results.append({**params, 'fold': fold_i + 1, 'macro_f1': macro_f1})
        print(f"Fold {fold_i+1} Macro F1: {macro_f1:.4f}")

    print(f"\n=== lr={params['learning_rate']} -> Media: {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f} ===")


--- lr0.0001_fold1 ---


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 79633.57it/s]
BertForSequenceClassification LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
bert.pooler.dense.weight                   | MISSING    | 
classifier.bias                            | MISSING    | 
bert.pooler.dense.bias                     | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECT

Baseline — entrenables: 592,130 / 109,852,418


/opt/miniconda3/envs/AP_FINAL/lib/python3.14/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


KeyboardInterrupt: 

## 4. Resumen y guardado

In [ ]:
df_folds = pd.DataFrame(all_fold_results)
df_folds.to_csv('results/cv_results_baseline.csv', index=False)

df_summary = (
    df_folds
    .groupby('learning_rate')['macro_f1']
    .agg(mean_macro_f1='mean', std_macro_f1='std')
    .reset_index()
    .sort_values('mean_macro_f1', ascending=False)
)
df_summary.to_csv('results/cv_results_baseline_summary.csv', index=False)
df_summary